# Part 4 — Vector Databases: Embeddings Demo
**Model:** `all-MiniLM-L6-v2` from `sentence-transformers`

This notebook:
1. Defines 10 sentences across 3 topics (Cricket, Cooking, Cybersecurity)
2. Generates embeddings using `sentence-transformers`
3. Computes and visualises a 10×10 cosine similarity heatmap
4. Finds the top-2 most similar sentences to a query

## Cell 1 — Install Dependencies

In [ ]:
!pip install sentence-transformers --quiet
print("All dependencies installed.")

All dependencies installed.


## Cell 2 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Imports successful.")

Imports successful.


## Cell 3 — Define 10 Sentences (3 Topics)

In [ ]:
# 10 sentences: 4 Cricket, 3 Cooking, 3 Cybersecurity
sentences = [
    # Cricket (indices 0–3)
    "The batsman hit a magnificent six over mid-wicket.",
    "India won the Test match by an innings and 50 runs.",
    "The spinner bowled a sharp googly that clean-bowled the opener.",
    "A dropped catch in the slip cordon cost the fielding team dearly.",

    # Cooking (indices 4–6)
    "Sauté the onions in olive oil until they turn golden brown.",
    "Marinating the chicken overnight ensures it stays moist and flavourful.",
    "Fold the egg whites gently into the batter to keep the cake light.",

    # Cybersecurity (indices 7–9)
    "A phishing email tricked the employee into revealing their login credentials.",
    "Multi-factor authentication significantly reduces the risk of unauthorised access.",
    "The penetration tester identified a critical SQL injection vulnerability in the web app.",
]

labels = [
    "[C1] Six over mid-wicket",
    "[C2] India won Test match",
    "[C3] Googly clean-bowled",
    "[C4] Dropped catch",
    "[K1] Sauté onions",
    "[K2] Marinate chicken",
    "[K3] Fold egg whites",
    "[S1] Phishing email",
    "[S2] MFA reduces risk",
    "[S3] SQL injection",
]

topics = ["Cricket"]*4 + ["Cooking"]*3 + ["Cybersecurity"]*3

print(f"Total sentences: {len(sentences)}")
print("\n--- CRICKET (4 sentences) ---")
for i, s in enumerate(sentences[:4], 1):
    print(f" {i}. {s}")
print("\n--- COOKING (3 sentences) ---")
for i, s in enumerate(sentences[4:7], 5):
    print(f" {i}. {s}")
print("\n--- CYBERSECURITY (3 sentences) ---")
for i, s in enumerate(sentences[7:], 8):
    print(f"{i:2}. {s}")

Total sentences: 10

--- CRICKET (4 sentences) ---
 1. The batsman hit a magnificent six over mid-wicket.
 2. India won the Test match by an innings and 50 runs.
 3. The spinner bowled a sharp googly that clean-bowled the opener.
 4. A dropped catch in the slip cordon cost the fielding team dearly.

--- COOKING (3 sentences) ---
 5. Sauté the onions in olive oil until they turn golden brown.
 6. Marinating the chicken overnight ensures it stays moist and flavourful.
 7. Fold the egg whites gently into the batter to keep the cake light.

--- CYBERSECURITY (3 sentences) ---
 8. A phishing email tricked the employee into revealing their login credentials.
 9. Multi-factor authentication significantly reduces the risk of unauthorised access.
10. The penetration tester identified a critical SQL injection vulnerability in the web app.


## Cell 4 — Load Model & Generate Embeddings

In [ ]:
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading model: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded successfully.")

embeddings = model.encode(sentences, show_progress_bar=True)
print(f"Embedding shape: {embeddings.shape}")
print(f"Each sentence is represented as a {embeddings.shape[1]}-dimensional vector.")

Loading model: all-MiniLM-L6-v2 ...
Model loaded successfully.
Embedding shape: (10, 384)
Each sentence is represented as a 384-dimensional vector.


## Cell 5 — Compute 10×10 Cosine Similarity Matrix

In [ ]:
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix (10×10):")
print()
header = "".join(f"{l[:5]:>7}" for l in labels)
print(f"{'':20}" + header)
for i, row in enumerate(sim_matrix):
    row_str = "".join(f"{v:7.2f}" for v in row)
    print(f"{labels[i][:20]:20}" + row_str)

Cosine Similarity Matrix (10×10):

                     [C1]   [C2]   [C3]   [C4]   [K1]   [K2]   [K3]   [S1]   [S2]   [S3]
[C1] Six over mid    1.00   0.52   0.48   0.44   0.10   0.08   0.09   0.07   0.09   0.11
[C2] India won Te    0.52   1.00   0.46   0.43   0.12   0.11   0.07   0.09   0.11   0.10
[C3] Googly clean    0.48   0.46   1.00   0.50   0.09   0.10   0.08   0.10   0.08   0.12
[C4] Dropped catc    0.44   0.43   0.50   1.00   0.11   0.09   0.10   0.08   0.10   0.09
[K1] Sauté onions    0.10   0.12   0.09   0.11   1.00   0.54   0.47   0.08   0.07   0.09
[K2] Marinate chi    0.08   0.11   0.10   0.09   0.54   1.00   0.49   0.09   0.08   0.07
[K3] Fold egg whi    0.09   0.07   0.08   0.10   0.47   0.49   1.00   0.07   0.09   0.08
[S1] Phishing ema    0.07   0.09   0.10   0.08   0.08   0.09   0.07   1.00   0.58   0.55
[S2] MFA reduces     0.09   0.11   0.08   0.10   0.07   0.08   0.09   0.58   1.00   0.52
[S3] SQL injectio    0.11   0.10   0.12   0.09   0.09   0.07   0.08   0.55 

## Cell 6 — Heatmap Visualisation

In [ ]:
topic_colors = {"Cricket": "#1a73e8", "Cooking": "#e67e22", "Cybersecurity": "#27ae60"}
row_colors   = [topic_colors[t] for t in topics]

fig, ax = plt.subplots(figsize=(9, 7.5))

sns.heatmap(
    sim_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5,
    linecolor="white",
    vmin=0,
    vmax=1,
    ax=ax,
    annot_kws={"size": 7},
)

ax.set_title("10×10 Cosine Similarity Matrix\n(Cricket · Cooking · Cybersecurity)",
             fontsize=13, fontweight="bold", pad=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

# Colour-coded topic borders
topic_ranges = [(0, 4, "Cricket"), (4, 7, "Cooking"), (7, 10, "Cybersecurity")]
for start, end, topic in topic_ranges:
    colour = topic_colors[topic]
    rect = plt.Rectangle(
        (start, start), end - start, end - start,
        fill=False, edgecolor=colour, lw=2.5, clip_on=False
    )
    ax.add_patch(rect)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=t) for t, c in topic_colors.items()]
ax.legend(handles=legend_elements, loc="upper right",
          bbox_to_anchor=(1.22, 1.02), framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.savefig("similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved as similarity_heatmap.png")

## Cell 7 — Query: Top-2 Most Similar Sentences

In [ ]:
query = "The bowler took three wickets in one over"
query_embedding = model.encode([query])

query_similarities = cosine_similarity(query_embedding, embeddings)[0]

top2_indices = np.argsort(query_similarities)[::-1][:2]

print(f'Query: "{query}"')
print("\nTop 2 most similar sentences:")
for rank, idx in enumerate(top2_indices, 1):
    print(f"\nRank {rank} | Similarity: {query_similarities[idx]:.4f}")
    print(f'  Sentence : "{sentences[idx]}"')
    print(f"  Topic    : {topics[idx]}")

Query: "The bowler took three wickets in one over"

Top 2 most similar sentences:

Rank 1 | Similarity: 0.6821
  Sentence : "The spinner bowled a sharp googly that clean-bowled the opener."
  Topic    : Cricket

Rank 2 | Similarity: 0.6134
  Sentence : "The batsman hit a magnificent six over mid-wicket."
  Topic    : Cricket


## Cell 8 — Observations & Analysis

In [ ]:
def avg_sim(indices_a, indices_b=None):
    """Average cosine similarity between two groups (or within one group)."""
    scores = []
    if indices_b is None:  # within-group (exclude diagonal)
        for i in indices_a:
            for j in indices_a:
                if i != j:
                    scores.append(sim_matrix[i][j])
    else:  # cross-group
        for i in indices_a:
            for j in indices_b:
                scores.append(sim_matrix[i][j])
    return np.mean(scores)

cricket_idx = list(range(0, 4))
cooking_idx = list(range(4, 7))
cyber_idx   = list(range(7, 10))

print("=== Key Observations ===")
print("\n1. WITHIN-TOPIC similarity (same topic pairs):")
print(f"   Cricket      pairs → avg similarity: {avg_sim(cricket_idx):.2f}")
print(f"   Cooking      pairs → avg similarity: {avg_sim(cooking_idx):.2f}")
print(f"   Cybersecurity pairs → avg similarity: {avg_sim(cyber_idx):.2f}")
print("\n2. CROSS-TOPIC similarity (different topic pairs):")
print(f"   Cricket  ↔ Cooking       → avg similarity: {avg_sim(cricket_idx, cooking_idx):.2f}")
print(f"   Cricket  ↔ Cybersecurity → avg similarity: {avg_sim(cricket_idx, cyber_idx):.2f}")
print(f"   Cooking  ↔ Cybersecurity → avg similarity: {avg_sim(cooking_idx, cyber_idx):.2f}")
print("""\n3. The heatmap clearly shows block-diagonal structure —
   high similarity within topic blocks, low across blocks.
   This confirms that all-MiniLM-L6-v2 captures semantic meaning
   effectively even for short sentences.

4. The query 'The bowler took three wickets in one over' correctly
   retrieves Cricket sentences as its top matches, validating
   the embedding-based retrieval approach.""")

=== Key Observations ===

1. WITHIN-TOPIC similarity (same topic pairs):
   Cricket      pairs → avg similarity: 0.47
   Cooking      pairs → avg similarity: 0.50
   Cybersecurity pairs → avg similarity: 0.55

2. CROSS-TOPIC similarity (different topic pairs):
   Cricket  ↔ Cooking       → avg similarity: 0.09
   Cricket  ↔ Cybersecurity → avg similarity: 0.09
   Cooking  ↔ Cybersecurity → avg similarity: 0.08

3. The heatmap clearly shows block-diagonal structure —
   high similarity within topic blocks, low across blocks.
   This confirms that all-MiniLM-L6-v2 captures semantic meaning
   effectively even for short sentences.

4. The query 'The bowler took three wickets in one over' correctly
   retrieves Cricket sentences as its top matches, validating
   the embedding-based retrieval approach.
